# Notebook 2 - Text Generation Models

This notebook trains and evaluates the **six text generation models** in the project:

| Model        | Character-level | Word-level |
|--------------|-----------------|------------|
| Markov Chain | order-5 n-gram  | order-2 n-gram |
| RNN          | Embedding + 2-layer RNN  + Linear | Embedding + 2-layer RNN  + Linear |
| LSTM         | Embedding + 2-layer LSTM + Linear | Embedding + 2-layer LSTM + Linear |

All code is imported from the `task1_text_generation/models/`, `shared/`, `task1_text_generation/train.py`, `task1_text_generation/evaluate.py` packages — this notebook is a narrated wrapper that trains the models, generates samples, and builds the comparison table from `evaluate_text_gen.py`.

> **Runtime note**: the full pipeline trains four neural nets for 20 epochs. On CPU this can take ~15–30 minutes. For a faster demo run, set `EPOCHS = 5` in the setup cell — the comparison still works, the quality is just lower.

## Setup

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 20  # drop to 5 for a fast demo run

print(f"Device: {DEVICE}")
print(f"Epochs: {EPOCHS}")

In [ ]:
corpus_path = os.path.join(PROJECT_ROOT, "task1_text_generation", "data", "corpus.txt")
with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = f.read()

print(f"Corpus: {len(corpus):,} chars, {len(corpus.split()):,} words")

## Part 1 - Markov Chain models

Markov chains are the baseline. They learn transition probabilities between n-gram states with no gradient descent — counting and normalizing only. Character-level uses order-5 (5 preceding characters), word-level uses order-2 (bigrams).

In [ ]:
from task1_text_generation.train import train_markov_char, train_markov_word

char_mcm, char_mcm_stats = train_markov_char(corpus, order=5)
word_mcm, word_mcm_stats = train_markov_word(corpus, order=2)

In [ ]:
print("--- Character-level Markov Chain samples ---\n")
for prompt in ["The ancient forest", "Science reveals", "The computer proc"]:
    out = char_mcm.generate_text(length=180, seed_text=prompt, mode="char", seed_value=7)
    print(f"[{prompt!r}]\n{out}\n")

print("\n--- Word-level Markov Chain samples ---\n")
for prompt in ["the ancient forest stretches", "the computer processes data", "many believe that the"]:
    out = word_mcm.generate_text(length=30, seed_text=prompt, mode="word", seed_value=7)
    print(f"[{prompt!r}]\n{out}\n")

## Part 2 - Neural models (RNN and LSTM)

We train four neural models: character- and word-level RNN and LSTM. All use:

- Embedding layer (`embed_dim=64`)
- 2 stacked recurrent layers (`hidden_dim=128`)
- Dropout `0.2`
- Linear output projection to vocabulary size
- Adam with StepLR, gradient clipping at max-norm 1.0

Character models use `seq_length=100`; word models use `seq_length=20` (since words carry more information per token).

The training helpers (`train_neural_model`) live in `task1_text_generation/train.py` and return `(model, tokenizer, stats)` — we store those so we can re-use the trained models for generation and comparison later.

In [ ]:
from task1_text_generation.models.rnn_model import RNNModel
from task1_text_generation.models.lstm_model import LSTMModel
from shared.tokenizer import CharTokenizer, WordTokenizer
from task1_text_generation.train import train_neural_model

### 2a. Character-level RNN

In [ ]:
char_tok_rnn = CharTokenizer()
char_tok_rnn.fit(corpus)

char_rnn = RNNModel(
    vocab_size=char_tok_rnn.vocab_size,
    embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.2,
)
char_rnn, char_tok_rnn, char_rnn_stats = train_neural_model(
    char_rnn, char_tok_rnn, corpus, mode="char",
    seq_length=100, batch_size=64, epochs=EPOCHS, lr=0.003,
    device=DEVICE, model_name="Char RNN",
)

### 2b. Word-level RNN

In [ ]:
word_tok_rnn = WordTokenizer()
word_tok_rnn.fit(corpus)

word_rnn = RNNModel(
    vocab_size=word_tok_rnn.vocab_size,
    embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.2,
)
word_rnn, word_tok_rnn, word_rnn_stats = train_neural_model(
    word_rnn, word_tok_rnn, corpus, mode="word",
    seq_length=20, batch_size=32, epochs=EPOCHS, lr=0.003,
    device=DEVICE, model_name="Word RNN",
)

### 2c. Character-level LSTM

In [ ]:
char_tok_lstm = CharTokenizer()
char_tok_lstm.fit(corpus)

char_lstm = LSTMModel(
    vocab_size=char_tok_lstm.vocab_size,
    embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.2,
)
char_lstm, char_tok_lstm, char_lstm_stats = train_neural_model(
    char_lstm, char_tok_lstm, corpus, mode="char",
    seq_length=100, batch_size=64, epochs=EPOCHS, lr=0.003,
    device=DEVICE, model_name="Char LSTM",
)

### 2d. Word-level LSTM

In [ ]:
word_tok_lstm = WordTokenizer()
word_tok_lstm.fit(corpus)

word_lstm = LSTMModel(
    vocab_size=word_tok_lstm.vocab_size,
    embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.2,
)
word_lstm, word_tok_lstm, word_lstm_stats = train_neural_model(
    word_lstm, word_tok_lstm, corpus, mode="word",
    seq_length=20, batch_size=32, epochs=EPOCHS, lr=0.003,
    device=DEVICE, model_name="Word LSTM",
)

## Part 3 - Loss curves

Per-epoch training loss for the four neural models. Markov chains are excluded since they are count-based and have no loss signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=False)

axes[0].plot(char_rnn_stats["losses"],  label="Char RNN",  marker="o", markersize=3)
axes[0].plot(char_lstm_stats["losses"], label="Char LSTM", marker="s", markersize=3)
axes[0].set_title("Character-level training loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(word_rnn_stats["losses"],  label="Word RNN",  marker="o", markersize=3, color="C2")
axes[1].plot(word_lstm_stats["losses"], label="Word LSTM", marker="s", markersize=3, color="C3")
axes[1].set_title("Word-level training loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("cross-entropy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4 - Comparison across all six models

We re-use `evaluate_text_gen_models` from `task1_text_generation/evaluate.py`, which runs 10 character-level prompts and 10 word-level prompts through every model and returns a formatted table. The helper expects a single `results` dict keyed by model name, so we assemble it here.

In [ ]:
from task1_text_generation.evaluate import evaluate_text_gen_models

results = {
    "char_mcm":  (char_mcm,  None,          char_mcm_stats),
    "word_mcm":  (word_mcm,  None,          word_mcm_stats),
    "char_rnn":  (char_rnn,  char_tok_rnn,  char_rnn_stats),
    "word_rnn":  (word_rnn,  word_tok_rnn,  word_rnn_stats),
    "char_lstm": (char_lstm, char_tok_lstm, char_lstm_stats),
    "word_lstm": (word_lstm, word_tok_lstm, word_lstm_stats),
}

report = evaluate_text_gen_models(results, device=DEVICE)

## Part 5 - Free-form sampling

Drop in any prompt here and compare what each architecture produces. Temperature controls how adventurous the neural models are — lower values stay closer to the most likely next token, higher values take more risks.

In [ ]:
CHAR_PROMPT = "The scientist discovered"
WORD_PROMPT = "the scientist discovered that"
TEMPERATURE = 0.8

print(f"--- Character-level, prompt={CHAR_PROMPT!r} ---\n")
print("Markov Chain:")
print(char_mcm.generate_text(length=160, seed_text=CHAR_PROMPT, mode="char", seed_value=1), "\n")
print("RNN:")
print(char_rnn.generate(char_tok_rnn, CHAR_PROMPT, length=120, temperature=TEMPERATURE, mode="char", device=DEVICE), "\n")
print("LSTM:")
print(char_lstm.generate(char_tok_lstm, CHAR_PROMPT, length=120, temperature=TEMPERATURE, mode="char", device=DEVICE), "\n")

print(f"\n--- Word-level, prompt={WORD_PROMPT!r} ---\n")
print("Markov Chain:")
print(word_mcm.generate_text(length=30, seed_text=WORD_PROMPT, mode="word", seed_value=1), "\n")
print("RNN:")
print(word_rnn.generate(word_tok_rnn, WORD_PROMPT, length=25, temperature=TEMPERATURE, mode="word", device=DEVICE), "\n")
print("LSTM:")
print(word_lstm.generate(word_tok_lstm, WORD_PROMPT, length=25, temperature=TEMPERATURE, mode="word", device=DEVICE), "\n")

## Takeaways

- **Markov chains** train in milliseconds and produce locally plausible text but have no real notion of long-range structure.
- **RNNs** learn smoother character- and word-level patterns than Markov, but vanishing gradients make longer contexts hard.
- **LSTMs** outperform RNNs at equal parameter budget, especially at the character level where longer dependencies matter more.
- The corpus is small (~12k words), so none of the neural models reach fluent English. Scaling the corpus is the main lever for better quality.

Next: `../../task2_chatbot/notebooks/02_chatbots.ipynb` will cover the seq2seq LSTM (with Bahdanau attention) and the from-scratch Transformer chatbot on the QA dataset.